# Phase 2A — Random Forest Baseline (Tabular Features)

**เป้าหมาย:** Train RandomForestClassifier บน tabular features (88 dims) จาก Phase 1B `.pt` files  
**Input:** Pre-extracted, StandardScaler-scaled tabular features — ใช้ได้เลย ไม่ต้อง scale ซ้ำ  
**Output:** `output/models/rf_baseline.pkl`, `rf_results.json`, confusion matrix plots, feature importance plot

| Split | Files | Note |
|---|---|---|
| training | 25,548 | 4 chunks/track → RF benefits naturally |
| validation | 798 | 1 chunk/track (center) |
| test | 800 | 1 chunk/track (center) — touch only once! |

In [ ]:
# --- Cell 1: Imports & Config ---
import json
import pickle
import time
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

from pathlib import Path

import numpy as np
import torch
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import RandomizedSearchCV
import matplotlib.pyplot as plt
import seaborn as sns

# Paths (relative to notebook location inside 'data prep/')
PT_DIR     = Path('output/pt_features')
MODELS_DIR = Path('output/models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

GENRE_LABELS = [
    'Electronic', 'Experimental', 'Folk', 'Hip-Hop',
    'Instrumental', 'International', 'Pop', 'Rock',
]
NUM_CLASSES = 8

# 88 tabular feature names (ตรงกับลำดับใน extract_tabular_features ของ Phase 1A)
FEATURE_NAMES = (
    [f'mfcc_mean_{i}' for i in range(20)] +
    [f'mfcc_std_{i}'  for i in range(20)] +
    ['centroid_mean', 'centroid_std'] +
    ['bandwidth_mean', 'bandwidth_std'] +
    ['rolloff_mean', 'rolloff_std'] +
    [f'chroma_mean_{i}' for i in range(12)] +
    [f'chroma_std_{i}'  for i in range(12)] +
    ['zcr_mean', 'zcr_std'] +
    ['rms_mean', 'rms_std'] +
    [f'contrast_mean_{i}' for i in range(7)] +
    [f'contrast_std_{i}'  for i in range(7)]
)
assert len(FEATURE_NAMES) == 88

print('Imports OK')
print(f'PT_DIR: {PT_DIR.resolve()}')
print(f'MODELS_DIR: {MODELS_DIR.resolve()}')

---
## Cell 2 — Load Tabular Features from .pt Files

In [ ]:
# --- Cell 2: Load tabular features ---
def load_split(split_name: str) -> tuple[np.ndarray, np.ndarray]:
    """
    Load tabular features + labels จากทุก .pt file ใน PT_DIR/split_name/
    Returns X (N, 88) float32, y (N,) int64
    """
    files = sorted((PT_DIR / split_name).glob('*.pt'))
    if not files:
        raise FileNotFoundError(f'No .pt files in {PT_DIR / split_name}')

    X, y = [], []
    for f in files:
        d = torch.load(f, weights_only=True)
        X.append(d['tabular'].to(torch.float32).numpy())
        y.append(int(d['label']))

    X_arr = np.stack(X)
    y_arr = np.array(y, dtype=np.int64)
    assert not np.isnan(X_arr).any(), f'NaN in {split_name}!'
    return X_arr, y_arr


print('Loading .pt files...')
t0 = time.time()
X_train, y_train = load_split('training')
X_val,   y_val   = load_split('validation')
X_test,  y_test  = load_split('test')
print(f'Load time: {time.time() - t0:.1f}s')

print(f'\nShapes:')
print(f'  X_train: {X_train.shape}   y_train: {y_train.shape}')
print(f'  X_val:   {X_val.shape}    y_val:   {y_val.shape}')
print(f'  X_test:  {X_test.shape}   y_test:  {y_test.shape}')
print(f'\nClass distribution (training):')
for i, g in enumerate(GENRE_LABELS):
    n = (y_train == i).sum()
    print(f'  {g:<15} {n:,}')

---
## Cell 3 — Train Random Forest

In [ ]:
# --- Cell 3: Train ---
# n_estimators=500: ดี balance ระหว่าง accuracy และ speed
# max_depth=None: grow fully — RF ป้องกัน overfit ด้วย bagging/feature subsampling
# class_weight='balanced': คำนึงถึง class imbalance (FMA เกือบ balanced อยู่แล้ว)
# n_jobs=-1: ใช้ทุก CPU core

rf = RandomForestClassifier(
    n_estimators     = 500,
    max_depth        = None,
    min_samples_leaf = 2,
    n_jobs           = -1,
    random_state     = 42,
    class_weight     = 'balanced',
)

print('Training RandomForestClassifier (n_estimators=500)...')
t0 = time.time()
rf.fit(X_train, y_train)
train_time = time.time() - t0
print(f'Done in {train_time:.1f}s')
print(f'OOB score not computed (oob_score=False for speed)')

---
## Cell 4 — Validation Evaluation

In [ ]:
# --- Cell 4: Validation evaluation ---
y_val_pred = rf.predict(X_val)
val_f1 = f1_score(y_val, y_val_pred, average='weighted')

print(f'Validation weighted F1: {val_f1:.4f}')
print()
print(classification_report(y_val, y_val_pred, target_names=GENRE_LABELS, digits=3))

---
## Cell 5 — Confusion Matrix (Validation)

In [ ]:
# --- Cell 5: Confusion matrix — validation ---
cm_val = confusion_matrix(y_val, y_val_pred)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    cm_val,
    annot        = True,
    fmt          = 'd',
    cmap         = 'Blues',
    xticklabels  = GENRE_LABELS,
    yticklabels  = GENRE_LABELS,
    ax           = ax,
)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('True', fontsize=12)
ax.set_title(f'Confusion Matrix — Validation (weighted F1={val_f1:.4f})', fontsize=13)
plt.tight_layout()
save_path = MODELS_DIR / 'rf_confusion_val.png'
fig.savefig(save_path, dpi=150)
plt.show()
print(f'Saved: {save_path}')

---
## Cell 6 — Hyperparameter Search (Optional, ~5 min)

> ข้ามได้ถ้าไม่ต้องการ tune — ผล default (n_estimators=500) มักดีอยู่แล้วสำหรับ RF

In [ ]:
# --- Cell 6: Hyperparameter search (optional) ---
RUN_SEARCH = False   # เปลี่ยนเป็น True เพื่อ run (~5-10 นาที)

if RUN_SEARCH:
    param_dist = {
        'n_estimators':     [100, 200, 300, 500, 700, 1000],
        'max_depth':        [10, 20, 30, None],
        'min_samples_leaf': [1, 2, 3, 5],
        'max_features':     ['sqrt', 'log2'],
    }
    base_rf = RandomForestClassifier(
        n_jobs=-1, random_state=42, class_weight='balanced'
    )
    search = RandomizedSearchCV(
        base_rf,
        param_distributions = param_dist,
        n_iter              = 20,
        cv                  = 3,
        scoring             = 'f1_weighted',
        n_jobs              = -1,
        random_state        = 42,
        verbose             = 1,
    )
    t0 = time.time()
    search.fit(X_train, y_train)
    print(f'Search time: {time.time() - t0:.1f}s')
    print(f'Best CV F1:  {search.best_score_:.4f}')
    print(f'Best params: {search.best_params_}')

    # Retrain with best params if meaningfully better
    y_best_val = search.best_estimator_.predict(X_val)
    best_val_f1 = f1_score(y_val, y_best_val, average='weighted')
    print(f'Best model val F1: {best_val_f1:.4f} (default: {val_f1:.4f})')
    if best_val_f1 > val_f1 + 0.005:   # > 0.5% improvement
        print('Using best estimator from search.')
        rf = search.best_estimator_
        val_f1 = best_val_f1
        y_val_pred = y_best_val
    else:
        print('Default params sufficient — keeping original RF.')
else:
    print('Hyperparameter search skipped (RUN_SEARCH=False).')

---
## Cell 7 — Test Evaluation (Run Once — Final!)

In [ ]:
# --- Cell 7: Test evaluation ---
# ⚠️ รัน cell นี้ครั้งเดียวเท่านั้น — test set ต้องไม่ใช้ระหว่าง development

y_test_pred = rf.predict(X_test)
test_f1 = f1_score(y_test, y_test_pred, average='weighted')

print(f'Test weighted F1: {test_f1:.4f}')
print()
print(classification_report(y_test, y_test_pred, target_names=GENRE_LABELS, digits=3))

# Confusion matrix — test
cm_test = confusion_matrix(y_test, y_test_pred)
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    cm_test,
    annot        = True,
    fmt          = 'd',
    cmap         = 'Oranges',
    xticklabels  = GENRE_LABELS,
    yticklabels  = GENRE_LABELS,
    ax           = ax,
)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('True', fontsize=12)
ax.set_title(f'Confusion Matrix — Test (weighted F1={test_f1:.4f})', fontsize=13)
plt.tight_layout()
save_path = MODELS_DIR / 'rf_confusion_test.png'
fig.savefig(save_path, dpi=150)
plt.show()
print(f'Saved: {save_path}')

---
## Cell 8 — Latency / RTF Benchmark

In [ ]:
# --- Cell 8: Latency benchmark ---
# จำลอง real-time inference: 1 sample (88 features) ต่อการ predict ครั้ง
# RTF = latency_ms / 3000ms  →  < 1.0 = real-time capable

sample = X_test[:1]
N_TRIALS = 1000

# Warm-up
for _ in range(10):
    rf.predict(sample)

times = []
for _ in range(N_TRIALS):
    t0 = time.perf_counter()
    rf.predict(sample)
    times.append(time.perf_counter() - t0)

latency_ms = float(np.mean(times) * 1000)
latency_p95 = float(np.percentile(times, 95) * 1000)
rtf = latency_ms / 3000.0

print(f'Single-sample latency (mean over {N_TRIALS} trials):')
print(f'  Mean: {latency_ms:.3f} ms')
print(f'  P95:  {latency_p95:.3f} ms')
print(f'  RTF:  {rtf:.6f}  ({"OK" if rtf < 1.0 else "FAIL"} — target < 1.0)')

---
## Cell 9 — Feature Importance (Top 20)

In [ ]:
# --- Cell 9: Feature importance ---
TOP_N = 20
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1][:TOP_N]
top_names = [FEATURE_NAMES[i] for i in indices]
top_vals  = importances[indices]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(range(TOP_N), top_vals[::-1], align='center', color='steelblue')
ax.set_yticks(range(TOP_N))
ax.set_yticklabels(top_names[::-1])
ax.set_xlabel('Mean Decrease in Impurity', fontsize=11)
ax.set_title(f'Random Forest — Top {TOP_N} Feature Importances', fontsize=13)
plt.tight_layout()
save_path = MODELS_DIR / 'rf_feature_importance.png'
fig.savefig(save_path, dpi=150)
plt.show()
print(f'Saved: {save_path}')

print(f'\nTop {TOP_N} features:')
for name, val in zip(top_names, top_vals):
    bar = '#' * int(val / top_vals[0] * 25)
    print(f'  {name:<20} {val:.4f}  {bar}')

---
## Cell 10 — Save Model & Summary

In [ ]:
# --- Cell 10: Save + summary ---

# Save model
model_path = MODELS_DIR / 'rf_baseline.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(rf, f)
print(f'Saved model: {model_path}  ({model_path.stat().st_size / 1e6:.1f} MB)')

# Save results
results = {
    'val_f1_weighted':  float(val_f1),
    'test_f1_weighted': float(test_f1),
    'per_class_f1': dict(zip(
        GENRE_LABELS,
        f1_score(y_test, y_test_pred, average=None).tolist(),
    )),
    'latency_ms':      latency_ms,
    'latency_p95_ms':  latency_p95,
    'rtf':             rtf,
    'n_estimators':    rf.n_estimators,
    'n_train_samples': int(X_train.shape[0]),
    'train_time_s':    round(train_time, 1),
}
results_path = MODELS_DIR / 'rf_results.json'
with open(results_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print(f'Saved results: {results_path}')

# Reload check
with open(model_path, 'rb') as f:
    rf2 = pickle.load(f)
reload_f1 = f1_score(y_test, rf2.predict(X_test), average='weighted')
assert abs(reload_f1 - test_f1) < 1e-6, 'Reload mismatch!'
print('Reload check: OK')

# Final summary
print()
print('=' * 60)
print('  PHASE 2A COMPLETE — Random Forest Baseline')
print('=' * 60)
print(f'  Val  weighted F1 : {val_f1:.4f}')
print(f'  Test weighted F1 : {test_f1:.4f}')
print(f'  Latency (mean)   : {latency_ms:.3f} ms')
print(f'  RTF              : {rtf:.6f}  (< 1.0 = real-time capable)')
print(f'  Training time    : {train_time:.1f}s')
print()
print('  Per-class F1 on test:')
per_class = f1_score(y_test, y_test_pred, average=None)
for genre, score in zip(GENRE_LABELS, per_class):
    bar = '#' * int(score * 30)
    print(f'    {genre:<15} {score:.3f}  {bar}')
print()
print('  Next: Phase 2B — Custom 2D CNN (mel spectrogram)')
print()
print('  Import for Phase 2B+:')
print('    from fma_phase2a_random_forest import load_split, GENRE_LABELS, FEATURE_NAMES')